# 국민은행_토스앱 리뷰 1년치 수집_저장_텍스트분석_mini_project
* 1. google play에서 국민은행 스타뱅킹앱, 토스앱의 사용자 리뷰를 1년치 수집
* 2. 수집한 리뷰를 mysql에 저장
* 3. 수집된 데이터로 긍정/부정 리뷰 비율 분석(막대 그래프 비교)
* 4. 워드클라우드 분석
* 5. LDA 토픽 모델링 분석(최적 k 탐색 포함)
* 6. word2vec 생성 후 T-sne 로 시각화
* 7. 분석 결과를 통해 얻을 수 있는 인사이트(문제점, 개선방안) 보고서 작성

In [2]:
# !pip install selenium webdriver-manager

## 데이터 수집 (토스, 국민은행)

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from datetime import datetime, timedelta
import pandas as pd
import time
from dbio import to_db

# Chrome 옵션 설정
options = Options()
options.add_argument("--window-size=1280,900")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# ChromeDriver 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.set_page_load_timeout(30)

apps = [
    ("https://play.google.com/store/apps/details?id=viva.republica.toss", "toss_reviews"),
    ("https://play.google.com/store/apps/details?id=com.kbstar.kbbank", "kbbank_reviews")
]

for url, table_name in apps:

    driver.get(url)
    print(driver.title)

    wait = WebDriverWait(driver, 10)

    # 자바스트립트로 윈도우를 0-1200px까지 스크롤 내리기
    driver.execute_script("window.scrollTo(0, 1200)")
    # 평점 및 리뷰 옆의 -> 버튼 찾아서 클릭하기
    btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="평점 및 리뷰 자세히 알아보기"]')))
    btn.click()
    # 최신순으로 리뷰를 정렬하기 위해 버튼 클릭
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
    button.click()
    time.sleep(5)
    # 최신을 찾아 클릭
    button2 = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span[aria-label="최신"]')))
    button2.click()
    time.sleep(5)
    ###########################################################
    # 팝업된 리뷰창에서 데이터 수집하기
    ###########################################################

    # 1년치 리뷰 수집하기
    start_date = datetime(2025, 3, 5)
    end_date = datetime(2026, 3, 5)

    while True:

        # 리뷰창 스크롤 내려서 과거 리뷰 로딩하기
        review_modal = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".fysCi.Vk3ZVd"))
        )
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", review_modal)
        time.sleep(1)
        # 리뷰 가장 마지막의 날짜 추출하기
        last_date = driver.find_elements(By.CSS_SELECTOR, ".bp9Aid")[-1].text
        last_date = last_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        last_date = datetime.strptime(last_date, "%Y-%m-%d")
        n_reviews = len(driver.find_elements(By.CSS_SELECTOR, ".bp9Aid"))
        print("리뷰개수:", n_reviews, "end_date", end_date, "last_date", last_date, end="\r")
        if last_date < start_date:
            break

    all_reviews = []    
    # 스크롤이 멈춘 후 데이터 수집하기
    for item in driver.find_elements(By.CSS_SELECTOR, ".RHo1pe"):
        result = {} 
        # 리뷰에서 날짜 추출하기
        date = item.find_element(By.CSS_SELECTOR, ".bp9Aid").text
        date = date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        review_date = datetime.strptime(date, "%Y-%m-%d")

        if not (start_date <= review_date <= end_date):
            continue

        # 평점 추출하기
        rating = item.find_element(By.CSS_SELECTOR, ".iXRFPc").get_attribute("aria-label")
        rating = rating.split()[3][0]
        # 리뷰 글 추출하기
        review_text = item.find_element(By.CSS_SELECTOR, ".h3YV2d").text

        all_reviews.append({"date": review_date, "rating": rating, "review_text": review_text})

    df = pd.DataFrame(all_reviews)
    df = df.drop_duplicates()
    display(df) 
    to_db("bank_app_reviews", table_name, df)

토스 - Google Play 앱


,date,rating,review_text
0,2026-03-05,1,용량 봐라...내 폰 자원 1기가 육박하게 잡아먹고 있네 ㅎ;;
1,2026-03-05,1,접속이 안됩니다. 특히 3일전부터 아예 안들어가지는 시간대가 너무 길고 잦아요. 다...
2,2026-03-05,1,업데이트하고 나서 왜 이체할 때 비밀번호 치는 창이 투명하게 뜨는 겁니까? 투명하게...
3,2026-03-05,3,폰트가 뭉게져서 나와요
4,2026-03-04,2,진짜 토스좋아하는데 요즘 에러도 자주나고 느려지고 아주 불편해요ㅜㅜㅜ
...,...,...,...
5835,2025-03-05,1,설치하고 나서 인증번호가 오지 않습니다. 서비스를 이용하지 못해요.......
5836,2025-03-05,3,갑자기 토스 앱에서 다른 계좌로 송금이 되지 않습니다 왜 그런 거죠 뱅킹에서는 아무...
5837,2025-03-05,5,홈화면 커스텀 해서 보고싶은 계좌 세팅 다해놨는데 뭔 자주보는 계좌니 뭐니 맘대로 ...
5838,2025-03-05,5,가족들 한테 추천받아 사용중 쪼와요


bank_app_reviews 데이터베이스 확인/생성 완료
bank_app_reviews.toss_reviews 데이터 저장 완료(append)
KB스타뱅킹-신분증, 결제, 통신도 다 되는 은행 - Google Play 앱


,date,rating,review_text
0,2026-03-05,5,잘사용하고있습니다 좋아요
1,2026-03-05,5,굳
2,2026-03-05,5,만족합니다
3,2026-03-05,5,편리하고 좋아요
4,2026-03-05,5,편안하다
...,...,...,...
4219,2025-03-05,5,구미공단점 직원도움으러잘가입함. 매우만족
4221,2025-03-05,1,비대면 뱅킹 가입 인증이 다른 메이저 금융어플 보다 훨씬 복잡함. 주민등록초본 발급...
4222,2025-03-05,5,정말 편해요~
4223,2025-03-05,5,Ui가 편하네요


bank_app_reviews 데이터베이스 확인/생성 완료
bank_app_reviews.kbbank_reviews 데이터 저장 완료(append)
